<a href="https://colab.research.google.com/github/deburky/language-models/blob/main/colab-eval-4bit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Eval: gpt-oss-claude-code (4-bit)

Loads `deburky/gpt-oss-claude-code` in NF4 4-bit via bitsandbytes (~10GB).  
Runs knowledge Q&A and tool-calling tasks, saves a markdown report.

Authored by: [Denis Burakov](https://www.github.com/deburky)

## 1. Install dependencies

In [1]:
%pip install transformers torch bitsandbytes accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 38.8 MB/s eta 0:00:00


In [2]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [3]:
!nvidia-smi

Fri Apr  3 17:52:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   28C    P0             46W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2. Load model in 4-bit

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = 'deburky/gpt-oss-claude-code'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='cuda',
)
print(f'Model loaded. GPU memory: {torch.cuda.memory_allocated() / 1e9:.1f} GB')

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/391 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/27.9M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/411 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/171 [00:00<?, ?B/s]

Model loaded. GPU memory: 40.9 GB


## 3. Helpers

In [5]:
import contextlib
import glob as globlib
import json
import pathlib
import re
import time
from datetime import datetime

KNOWLEDGE_PROMPTS = [
    'Who invented the method of Maximum Likelihood Estimation?',
    'Who is Alan Turing and what is he famous for?',
    'What is Weight of Evidence (WoE) in credit scoring and how is it calculated?',
    'Explain the difference between L1 and L2 regularization.',
    'What is the purpose of a chat template in large language models?',
]

TOOL_PROMPTS = [
    'List all Python files in the current directory.',
    'Read the first 20 lines of /etc/hostname.',
    'Search for the word root in /etc/hostname.',
]

TOOL_SYSTEM = """You are a coding assistant with access to the file system.

Available tools:
- read(path, offset, limit): Read a file. Use limit to cap lines returned.
- glob(pat): Find files by pattern
- grep(pat, path): Search for text in files

To use a tool, format it EXACTLY like this:
<tool_call>{"tool": "name", "args": {"key": "value"}}</tool_call>

When asked to read the first N lines, always pass limit: N.
Reply ONLY with a tool call or a final answer. Do NOT call more tools after you have the answer."""


def strip_gpt_oss(text):
    if '<|channel|>final<|message|>' in text:
        text = text.split('<|channel|>final<|message|>')[-1]
    return re.sub(r'<\|[^>]+\|>', '', text).strip()


def generate(messages, max_new_tokens=256):
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True,
        return_tensors='pt', return_dict=True,
    ).to(model.device)
    t0 = time.perf_counter()
    with torch.no_grad():
        output_ids = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            temperature=0.3, do_sample=True,
        )
    elapsed = time.perf_counter() - t0
    raw = tokenizer.decode(output_ids[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=False)
    # Truncate at first complete tool call
    for tag in ['</tool_call>', '</read>', '</glob>', '</grep>']:
        idx = raw.find(tag)
        if idx != -1:
            raw = raw[:idx + len(tag)]
            break
    return strip_gpt_oss(raw), elapsed


def parse_tool_calls(text):
    calls = []
    # Native gpt-oss format: <read>{...}</read>
    for tag in ['read', 'glob', 'grep']:
        for m in re.finditer(rf'<{tag}>(.*?)</{tag}>', text, re.DOTALL):
            with contextlib.suppress(Exception):
                calls.append({'name': tag, 'args': json.loads(m.group(1))})
    # <tool_call> format
    for m in re.finditer(r'<tool_call>\s*(\{.*?\})\s*</tool_call>', text, re.DOTALL):
        with contextlib.suppress(Exception):
            d = json.loads(m.group(1))
            if d.get('tool') in ('read', 'glob', 'grep'):
                calls.append({'name': d['tool'], 'args': d.get('args', {})})
    return calls


def run_tool(name, args):
    try:
        if name == 'glob':
            files = globlib.glob(args.get('pat', '*'), recursive=True)
            return '\n'.join(sorted(files)) or 'none'
        if name == 'read':
            lines = pathlib.Path(args['path']).read_text(errors='replace').splitlines()
            offset = int(args.get('offset', 0))
            limit = int(args.get('limit', 20))
            return ''.join(f'{offset+i+1:4}| {l}\n' for i, l in enumerate(lines[offset:offset+limit]))
        if name == 'grep':
            pat = re.compile(args.get('pat', ''))
            p = pathlib.Path(args.get('path', '.'))
            hits = []
            for fp in ([p] if p.is_file() else p.rglob('*')):
                if fp.is_file():
                    with contextlib.suppress(Exception):
                        for i, line in enumerate(fp.read_text(errors='replace').splitlines(), 1):
                            if pat.search(line):
                                hits.append(f'{fp}:{i}:{line}')
            return '\n'.join(hits[:40]) or 'none'
    except Exception as e:
        return f'error: {e}'
    return f'error: unknown tool {name}'


print('Helpers ready.')

Helpers ready.


## 4. Run evaluation

In [6]:
knowledge_results = []
tool_results = []

print('[Knowledge Q&A]')
for prompt in KNOWLEDGE_PROMPTS:
    print(f'  Q: {prompt[:60]}...')
    messages = [
        {'role': 'system', 'content': 'You are a knowledgeable assistant. Answer concisely and accurately.'},
        {'role': 'user', 'content': prompt},
    ]
    answer, latency = generate(messages, max_new_tokens=256)
    knowledge_results.append({'prompt': prompt, 'answer': answer, 'latency': latency})
    print(f'  {latency:.1f}s — {answer[:100]}')

print('\n[Tool Calling]')
for task in TOOL_PROMPTS:
    print(f'  T: {task}')
    messages = [
        {'role': 'system', 'content': TOOL_SYSTEM},
        {'role': 'user', 'content': task},
    ]
    tool_log = []
    total_time = 0.0
    for _ in range(5):
        response, elapsed = generate(messages, max_new_tokens=256)
        total_time += elapsed
        calls = parse_tool_calls(response)
        display = re.sub(r'<tool_call>.*?</tool_call>|<(?:read|glob|grep)>.*?</(?:read|glob|grep)>', '', response, flags=re.DOTALL).strip()
        tool_log.append({'response': display, 'tool_calls': []})
        if not calls:
            break
        for call in calls:
            result = run_tool(call['name'], call['args'])
            tool_log[-1]['tool_calls'].append({'tool': call['name'], 'args': call['args'], 'result': result[:400]})
            messages.append({'role': 'assistant', 'content': response})
            messages.append({'role': 'user', 'content': f"Tool {call['name']} returned:\n{result}"})
    tool_results.append({'task': task, 'log': tool_log, 'latency': total_time})
    print(f'{total_time:.1f}s — {len(tool_log)} step(s)')

print('\nDone.')

[Knowledge Q&A]
  Q: Who invented the method of Maximum Likelihood Estimation?...
  2.5s — Maximum likelihood estimation (MLE) was introduced by Ronald Fisher in the early 20th century. Fishe
  Q: Who is Alan Turing and what is he famous for?...
  2.4s — Alan Turing (1912–1954) was a British mathematician, logician, and computer scientist. He is famous 
  Q: What is Weight of Evidence (WoE) in credit scoring and how i...
  5.2s — Weight of Evidence (WoE) is a transformation technique used in credit scoring to convert categorical
  Q: Explain the difference between L1 and L2 regularization....
  1.4s — L1 regularization adds the absolute value of the coefficients to the loss function, encouraging spar
  Q: What is the purpose of a chat template in large language mod...
  0.8s — A chat template defines the structure of the conversation, including roles, formatting, and context 

[Tool Calling]
  T: List all Python files in the current directory.
0.9s — 2 step(s)
  T: Read the first 20 li

## 5. Save report

In [7]:
def fmt_tool_log(log):
    lines = []
    for step in log:
        if step['response']:
            lines.append(step['response'])
        for tc in step['tool_calls']:
            lines.append(f"\n**→ {tc['tool']}({json.dumps(tc['args'])})**")
            lines.append(f"```\n{tc['result']}\n```")
    return '\n'.join(lines).strip()


now = datetime.now().strftime('%Y-%m-%d %H:%M')
lines = [
    '# Evaluation Report',
    '',
    f'**Model:** `{MODEL_ID}` (4-bit NF4)  ',
    f'**Date:** {now}',
    '',
    '---',
    '',
    '## 1. Knowledge Q&A',
    '',
    '| Question | Answer | Latency |',
    '|---|---|---|',
]
for r in knowledge_results:
    ans = r['answer'].replace('\n', '<br>').replace('|', '\\|')
    lines.append(f"| {r['prompt']} | {ans} | {r['latency']:.1f}s |")

lines += ['', '---', '', '## 2. Tool Calling', '']
for r in tool_results:
    lines += [
        f"### {r['task']}",
        '',
        f"**Steps:** {len(r['log'])}  **Latency:** {r['latency']:.1f}s",
        '',
        fmt_tool_log(r['log']),
        '',
    ]

report = '\n'.join(lines)
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
out_path = f'eval_4bit_{ts}.md'
with open(out_path, 'w') as f:
    f.write(report)
print(f'Report saved: {out_path}')
print(report[:600])

Report saved: eval_4bit_20260403_175625.md
# Evaluation Report

**Model:** `deburky/gpt-oss-claude-code` (4-bit NF4)  
**Date:** 2026-04-03 17:56

---

## 1. Knowledge Q&A

| Question | Answer | Latency |
|---|---|---|
| Who invented the method of Maximum Likelihood Estimation? | Maximum likelihood estimation (MLE) was introduced by Ronald Fisher in the early 20th century. Fisher published the first formal description of MLE in his 1922 paper "On the Probable Error of a Regression," and further developed the theory in his 1925 book "Statistical Methods for Research Workers." Fisher is widely credited as the inventor of MLE. | 2.5s |
| 
